In [12]:
import joblib
import pandas as pd

# import the fraud detector pipeline
model_path = "../code/custom-classifier-model/fraud_detector.pkl"
model = joblib.load(model_path)

In [13]:
def analyze_data(user_input_data):
    """
    Analyze a single transaction and print whether fraud is detected.

    Parameters
    ----------
    user_input_data : dict or pandas.DataFrame
        - A dict mapping column names to values or single-element lists, OR
        - A single-row DataFrame with the same columns used in training.
    """
    # ensure dataframe exists
    if isinstance(user_input_data, dict):
        # if values are scalars, wrap them in a single row
        if not any(isinstance(v, (list, tuple, pd.Series)) for v in user_input_data.values()):
            user_df = pd.DataFrame([user_input_data])
        else:
            user_df = pd.DataFrame(user_input_data)
    elif isinstance(user_input_data, pd.Series):
        user_df = user_input_data.to_frame().T
    else:
        user_df = user_input_data

    # select only the columns expected by the model
    expected_columns = [
        'step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg',
        'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud'
    ]
    user_df = user_df[expected_columns]

    # let the loaded pipeline handle preprocessing and prediction
    prediction = model.predict(user_df)

    # display the result
    if prediction[0] == 1:
        print("Financial fraud is detected.")
    else:
        print("No financial fraud detected.")

    return prediction[0]

In [14]:
# example user input matching the models expected features
user_input_data = {
    'step': 1,
    'type': 'PAYMENT',
    'amount': 1000.0,
    'nameOrig': 'C123456789',
    'oldbalanceOrg': 5000.0,
    'newbalanceOrig': 4000.0,
    'nameDest': 'M123456789',
    'oldbalanceDest': 0.0,
    'newbalanceDest': 0.0,
    'isFlaggedFraud': 0,
}

# call the analyze_data function
analyze_data(user_input_data)

No financial fraud detected.


np.int64(0)

In [15]:

# simple test cases

# not fraud
non_fraud_tx = {
    'step': 1,
    'type': 'PAYMENT',
    'amount': 100.0,
    'nameOrig': 'C000000001',
    'oldbalanceOrg': 1000.0,
    'newbalanceOrig': 900.0,
    'nameDest': 'M000000001',
    'oldbalanceDest': 0.0,
    'newbalanceDest': 0.0,
    'isFlaggedFraud': 0,
}

print("Non-fraud test case:")
non_fraud_pred = analyze_data(non_fraud_tx)
print(f"Model output (0 = no fraud, 1 = fraud): {non_fraud_pred}\n")

# 2. fraud transfer with large amount and zero resulting balance
fraud_like_tx = {
    'step': 1,
    'type': 'TRANSFER',
    'amount': 250000.0,
    'nameOrig': 'C000000002',
    'oldbalanceOrg': 250000.0,
    'newbalanceOrig': 0.0,
    'nameDest': 'C000000003',
    'oldbalanceDest': 0.0,
    'newbalanceDest': 0.0,
    'isFlaggedFraud': 0,
}

print("Fraud-like test case:")
fraud_like_pred = analyze_data(fraud_like_tx)
print(f"Model output (0 = no fraud, 1 = fraud): {fraud_like_pred}")

fraud_like_tx2 = {
    'step': 1,
    'type': 'TRANSFER',
    'amount': 77777777777.00,
    'nameOrig': '67',
    'oldbalanceOrg': 0.0,
    'newbalanceOrig': 77777777777.00,
    'nameDest': '68',
    'oldbalanceDest': 77777777777.0,
    'newbalanceDest': 0.0,
    'isFlaggedFraud': 0,
}
print("Fraud-like test case2:")
fraud_like_pred2 = analyze_data(fraud_like_tx2)
print(f"Model output (0 = no fraud, 1 = fraud): {fraud_like_pred2}")


Non-fraud test case:
No financial fraud detected.
Model output (0 = no fraud, 1 = fraud): 0

Fraud-like test case:
No financial fraud detected.
Model output (0 = no fraud, 1 = fraud): 0
Fraud-like test case2:
No financial fraud detected.
Model output (0 = no fraud, 1 = fraud): 0


In [16]:
'''
How to use:
1-Locate the input_data.txt file in the folder.
2-Find the data you want to check for fraud on.
3-Enter that data onto input_data.txt. To test for fraud on multiple individual cases of data, separate each line of data with a new line.
3.1-Data is formatted as <step>,<type>,<amount>,<nameOrig>,<oldbalanceOrg>,<newbalanceOrig>,<nameDest>,<oldbalanceDest>,<newbalanceDest>,<isFlaggedFraud>
3.2-<isFlaggedFraud> must be a 0 or a 1. 0 if it is not flagged as fraud, 1 if it is flagged as fraud.
3.2.1-Example of a correctly formatted input: 1,TRANSFER,181.00,C2048537720,170136.0,160296.36,C553264065,0.0,0.0,0
4-Run the program and wait for it to finish running.
5-Locate and open the output_analysis.txt file. This file prints out the given data, shows if it is fraud or not, and explains why.
'''

'\nHow to use:\n1-Locate the input_data.txt file in the folder.\n2-Find the data you want to check for fraud on.\n3-Enter that data onto input_data.txt. To test for fraud on multiple individual cases of data, separate each line of data with a new line.\n3.1-Data is formatted as <step>,<type>,<amount>,<nameOrig>,<oldbalanceOrg>,<newbalanceOrig>,<nameDest>,<oldbalanceDest>,<newbalanceDest>,<isFlaggedFraud>\n3.2-<isFlaggedFraud> must be a 0 or a 1. 0 if it is not flagged as fraud, 1 if it is flagged as fraud.\n3.2.1-Example of a correctly formatted input: 1,TRANSFER,181.00,C2048537720,170136.0,160296.36,C553264065,0.0,0.0,0\n4-Run the program and wait for it to finish running.\n5-Locate and open the output_analysis.txt file. This file prints out the given data, shows if it is fraud or not, and explains why.\n'

In [17]:
# process input_data.txt and write predictions to output_analysis.txt
input_filepath = 'input_data.txt'
output_filepath = 'output_analysis.txt'

# Define expected columns in input file
columns = ['step','type','amount','nameOrig','oldbalanceOrg','newbalanceOrig','nameDest','oldbalanceDest','newbalanceDest','isFlaggedFraud']

records = []
with open(input_filepath, 'r', encoding='utf-8') as f:
    for line_num, line in enumerate(f, start=1):
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        parts = [x.strip() for x in line.split(',')]
        if len(parts) != len(columns):
            print(f"Skipping line {line_num}: expected {len(columns)} values, got {len(parts)}")
            continue
        record = dict(zip(columns, parts))
        # cast numeric fields
        try:
            record['step'] = int(record['step'])
            record['amount'] = float(record['amount'])
            record['oldbalanceOrg'] = float(record['oldbalanceOrg'])
            record['newbalanceOrig'] = float(record['newbalanceOrig'])
            record['oldbalanceDest'] = float(record['oldbalanceDest'])
            record['newbalanceDest'] = float(record['newbalanceDest'])
            record['isFlaggedFraud'] = int(record['isFlaggedFraud'])
            if record['isFlaggedFraud'] not in (0, 1):
                raise ValueError('isFlaggedFraud must be 0 or 1')
        except Exception as e:
            print(f"Skipping line {line_num} due to parse error: {e}")
            continue
        records.append(record)

if not records:
    raise ValueError('No valid records found in input_data.txt')

input_df = pd.DataFrame(records)

# run predictions for each record
output_rows = []
for idx, row in input_df.iterrows():
    pred = analyze_data(row)
    reason = 'Model predicts fraud' if pred == 1 else 'Model predicts non-fraud'
    output_rows.append({
        'line': idx+1,
        'step': row['step'],
        'type': row['type'],
        'amount': row['amount'],
        'isFlaggedFraud': row['isFlaggedFraud'],
        'prediction': int(pred),
        'reason': reason,
    })

# write analysis output
with open(output_filepath, 'w', encoding='utf-8') as out:
    out.write('line,step,type,amount,isFlaggedFraud,prediction,reason\n')
    for r in output_rows:
        out.write(f"{r['line']},{r['step']},{r['type']},{r['amount']},{r['isFlaggedFraud']},{r['prediction']},{r['reason']}\n")

print(f'Written {len(output_rows)} records to {output_filepath}')

No financial fraud detected.
No financial fraud detected.
Financial fraud is detected.
Financial fraud is detected.
No financial fraud detected.
Written 5 records to output_analysis.txt
